In [ ]:
#@title 1. GitHub source and authorization
REPO = "university-of-toronto-humanoid-robotics/humanoid" #@param {type:"string"}
BRANCH = "kartik/rpl" #@param {type:"string"}

from pathlib import Path
import os
import shutil
import subprocess
import sys
from IPython.display import HTML, display

REPO_DIR = Path("/content/humanoid")
PYTHON = "/content/mjlab-venv/bin/python"
if shutil.which("gh") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "gh"], check=True)
git_env = {**os.environ, "GH_BROWSER": "echo", "GH_PROMPT_DISABLED": "1", "GIT_TERMINAL_PROMPT": "0"}
git_env.pop("GH_TOKEN", None)
git_env.pop("GITHUB_TOKEN", None)
if subprocess.run(["gh", "auth", "status", "--hostname", "github.com"], env=git_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
    display(HTML('<a href="https://github.com/login/device" target="_blank">Open GitHub authorization</a><p>Enter the code below and approve access. This cell waits for you.</p>'))
    login = subprocess.Popen(["gh", "auth", "login", "--web", "--hostname", "github.com", "--git-protocol", "https"], env=git_env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in login.stdout:
            print(line, end="", flush=True)
        if login.wait():
            raise RuntimeError("GitHub login failed. Check the output above.")
    except KeyboardInterrupt:
        login.send_signal(2)
        login.wait()
        raise
subprocess.run(["gh", "auth", "setup-git", "--hostname", "github.com"], env=git_env, check=True)
url = f"https://github.com/{REPO}.git"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO_DIR)], env=git_env, check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    assert origin.lower().removesuffix(".git") == url.lower().removesuffix(".git"), "Existing checkout uses a different repository; use a fresh runtime."
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], env=git_env, check=True)
    current = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
    if current != BRANCH:
        subprocess.run(["git", "-C", str(REPO_DIR), "switch", BRANCH], env=git_env, check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive", "training/unitree_rl_mjlab"], env=git_env, check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=True)


In [ ]:
# 2. Install in an isolated venv using the same pins as the team's Docker setup.
# The requirements filename says native, but pip selects Linux/CUDA wheels here.
import time
assert (3, 10) <= sys.version_info[:2] < (3, 14), "Use Python 3.10–3.13 with these pins."
# Colab's Python version may not have a matching apt venv package.
venv_dir = Path(PYTHON).parent.parent
venv_ready = Path(PYTHON).is_file() and subprocess.run([PYTHON, "-m", "pip", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not venv_ready:
    print("Creating the Python environment...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", sys.executable, "--seed", "--clear", str(venv_dir)], check=True)
# RSL-RL passes start_method to W&B; W&B 0.29 removed that setting.
print("Installing training dependencies; pip output appears below.", flush=True)
install = subprocess.Popen(
    [PYTHON, "-u", "-m", "pip", "install", "-v", "--progress-bar", "raw",
     "-r", str(REPO_DIR / "docs/requirements-native-mjlab.txt"), "wandb<0.29"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
started = time.monotonic()
try:
    for line in install.stdout:
        print(line, end="", flush=True)
    return_code = install.wait()
except KeyboardInterrupt:
    install.terminate()
    install.wait()
    raise
if return_code:
    raise subprocess.CalledProcessError(return_code, install.args)
print(f"Dependencies installed in {time.monotonic() - started:.0f}s.", flush=True)


In [ ]:
# 3. Mount Drive and verify CUDA for both PyTorch and Warp.
from google.colab import drive
drive.mount("/content/drive")
WORK_DIR = Path("/content/drive/MyDrive/humanoid")
WORK_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([PYTHON, "-c", "import torch, warp as wp; wp.init(); assert torch.cuda.is_available() and wp.is_cuda_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name(0)); wp.zeros(1, device='cuda:0'); wp.synchronize_device('cuda:0')"], check=True)
print("Outputs will be saved under:", WORK_DIR / "logs/rsl_rl")


In [ ]:
# 4. Pick a task from this branch, set training options, then choose a checkpoint.
import json
import ipywidgets as widgets
from IPython.display import display

registry_code = (
    "import json, _setup, src.tasks; "
    "from mjlab.tasks.registry import list_tasks, load_env_cfg, load_rl_cfg; "
    "print('TASK_REGISTRY_JSON=' + json.dumps({name: "
    "{'experiment': load_rl_cfg(name).experiment_name, "
    "'start_level': 'start_terrain_level' in load_env_cfg(name).events} "
    "for name in list_tasks()}))"
)
registry = subprocess.run(
    [PYTHON, "-c", registry_code],
    cwd=REPO_DIR / "training/scripts",
    env={**os.environ, "G1_TRAIN_START_LEVEL": "0"},
    capture_output=True, text=True,
)
if registry.returncode:
    print(registry.stdout, registry.stderr)
    registry.check_returncode()
task_metadata = json.loads(next(
    line.removeprefix("TASK_REGISTRY_JSON=")
    for line in reversed(registry.stdout.splitlines())
    if line.startswith("TASK_REGISTRY_JSON=")
))
assert task_metadata, "No training tasks were registered by this branch."
print(f"Found {len(task_metadata)} tasks in {BRANCH}.")

style = {"description_width": "initial"}
task_widget = widgets.Dropdown(
    options=sorted(task_metadata),
    value="G1-Open-HeightMap" if "G1-Open-HeightMap" in task_metadata else sorted(task_metadata)[0],
    description="TASK", style=style,
)
start_level_widget = widgets.IntText(value=-1, description="START_TERRAIN_LEVEL", style=style)
num_envs_widget = widgets.BoundedIntText(value=1024, min=1, description="NUM_ENVS", style=style)
iterations_widget = widgets.BoundedIntText(value=2000, min=1, description="ITERATIONS", style=style)
rollout_steps_widget = widgets.BoundedIntText(value=24, min=1, description="ROLLOUT_STEPS", style=style)
minibatches_widget = widgets.BoundedIntText(value=4, min=1, description="NUM_MINIBATCHES", style=style)
save_interval_widget = widgets.BoundedIntText(value=200, min=1, description="SAVE_INTERVAL", style=style)
run_name_widget = widgets.Text(value="experiment_1", description="RUN_NAME", style=style)
wandb_project_widget = widgets.Text(value="humanoid-locomotion", description="WANDB_PROJECT", style=style)
wandb_entity_widget = widgets.Text(value="", description="WANDB_ENTITY", style=style)
option_panel = widgets.GridBox(
    [task_widget, start_level_widget, num_envs_widget, iterations_widget,
     rollout_steps_widget, minibatches_widget, save_interval_widget,
     run_name_widget, wandb_project_widget, wandb_entity_widget],
    layout=widgets.Layout(grid_template_columns="repeat(3, minmax(250px, 1fr))", grid_gap="10px"),
)

LOG_ROOT = WORK_DIR / "logs/rsl_rl"
HEIGHTMAP_TASKS = {
    name for name in task_metadata
    if name.startswith("RPL-") or (name.startswith("G1-") and name.endswith("-HeightMap"))
}
HEIGHTMAP_EXPERIMENTS = {task_metadata[name]["experiment"] for name in HEIGHTMAP_TASKS}
bundled_models = sorted({
    *(REPO_DIR / "training/models").glob("best_g1_*_heightmap/model_*.pt"),
    *(REPO_DIR / "training/models/g1_open_heightmap").glob("model_*.pt"),
    *(REPO_DIR / "training/models").glob("rpl_*/model_*.pt"),
})
flat_model = REPO_DIR / "training/models/best_g1_flat_heightmap/model_1999.pt"
rough_model = REPO_DIR / "training/models/best_g1_rough_heightmap/model_5996.pt"
open_model = REPO_DIR / "training/models/g1_open_heightmap/model_6995.pt"

resume_mode = widgets.Checkbox(
    value=task_widget.value in {"G1-Open-HeightMap", "G1-Open-Maze-HeightMap"},
    description="Start from checkpoint",
)
source_location = widgets.Dropdown(
    options=[("Repository model", "repo"), ("Drive run", "drive")],
    value="repo" if task_widget.value in HEIGHTMAP_TASKS else "drive",
    description="Checkpoint source:", style=style,
)
repo_model = widgets.Dropdown(
    options=[(str(p.relative_to(REPO_DIR)), str(p)) for p in bundled_models],
    description="Repository model:", style=style,
    layout=widgets.Layout(width="95%"),
)
source_environment = widgets.Dropdown(description="Source env:", style=style)
source_run = widgets.Dropdown(description="Source run:", layout=widgets.Layout(width="95%"))
source_model = widgets.Dropdown(description="Model:")
repo_browser = widgets.VBox([repo_model])
drive_browser = widgets.VBox([source_environment, source_run, source_model])
checkpoint_browser = widgets.VBox([source_location, repo_browser, drive_browser])

def update_models(change=None):
    folder = LOG_ROOT / source_environment.value / source_run.value if source_environment.value and source_run.value else None
    models = list(folder.glob("model_*.pt")) if folder else []
    models.sort(key=lambda p: (int(p.stem[6:]) if p.stem[6:].isdigit() else -1, p.name), reverse=True)
    source_model.options = [(p.name, str(p)) for p in models]
    source_model.value = str(models[0]) if models else None

def update_runs(change=None):
    folder = LOG_ROOT / source_environment.value if source_environment.value else None
    source_run.options = sorted(
        (p.name for p in folder.iterdir() if p.is_dir() and any(p.glob("model_*.pt"))),
        reverse=True,
    ) if folder else []
    source_run.value = source_run.options[0] if source_run.options else None
    update_models()

def update_mode(change=None):
    checkpoint_browser.layout.display = "" if resume_mode.value else "none"
    repo_browser.layout.display = "" if source_location.value == "repo" else "none"
    drive_browser.layout.display = "" if source_location.value == "drive" else "none"

def update_task(change=None):
    compatible = task_widget.value in HEIGHTMAP_TASKS
    folders = (p for p in LOG_ROOT.iterdir() if p.is_dir() and any(p.glob("*/model_*.pt"))) if LOG_ROOT.exists() else ()
    source_environment.options = sorted(
        p.name for p in folders if (p.name in HEIGHTMAP_EXPERIMENTS) == compatible
    )
    source_environment.value = source_environment.options[0] if source_environment.options else None
    update_runs()
    source_location.value = "repo" if compatible and bundled_models else "drive"
    resume_mode.value = task_widget.value.startswith("RPL-") or task_widget.value in {
        "G1-Open-HeightMap", "G1-Open-Maze-HeightMap"
    }
    if task_widget.value.startswith("RPL-") and flat_model in bundled_models:
        repo_model.value = str(flat_model)
    elif task_widget.value.startswith("G1-Open-") and open_model in bundled_models:
        repo_model.value = str(open_model)
    elif rough_model in bundled_models:
        repo_model.value = str(rough_model)
    update_mode()

source_environment.observe(update_runs, names="value")
source_run.observe(update_models, names="value")
source_location.observe(update_mode, names="value")
resume_mode.observe(update_mode, names="value")
task_widget.observe(update_task, names="value")
update_task()
display(option_panel, resume_mode, checkpoint_browser)
print("Iterations are additional updates when resuming. Repository models require a compatible height-map task.")


In [ ]:
# 5. Train with W&B logging. Use Colab's stop button to interrupt.
TASK = task_widget.value
START_TERRAIN_LEVEL = start_level_widget.value
NUM_ENVS = num_envs_widget.value
ITERATIONS = iterations_widget.value
ROLLOUT_STEPS = rollout_steps_widget.value
NUM_MINIBATCHES = minibatches_widget.value
SAVE_INTERVAL = save_interval_widget.value
RUN_NAME = run_name_widget.value.strip()
WANDB_PROJECT = wandb_project_widget.value.strip()
WANDB_ENTITY = wandb_entity_widget.value.strip()
assert TASK in task_metadata, "Select a registered task in cell 4."
assert all(n > 0 for n in (NUM_ENVS, ITERATIONS, ROLLOUT_STEPS, NUM_MINIBATCHES, SAVE_INTERVAL))
assert (NUM_ENVS * ROLLOUT_STEPS) % NUM_MINIBATCHES == 0, "Rollout size must divide into minibatches."
assert RUN_NAME and WANDB_PROJECT, "Enter a run name and W&B project."
assert START_TERRAIN_LEVEL == -1 or (
    0 <= START_TERRAIN_LEVEL <= 9 and task_metadata[TASK]["start_level"]
), "This task does not support the selected start terrain level."
RESUME_CHECKPOINT = (
    (repo_model.value if source_location.value == "repo" else source_model.value)
    if resume_mode.value else ""
)
if resume_mode.value:
    assert RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).is_file(), "Select an existing checkpoint in cell 4."
    if source_location.value == "repo":
        assert TASK in HEIGHTMAP_TASKS, "Repository models require a compatible height-map task."
print(f"Training {TASK} from {RESUME_CHECKPOINT or 'scratch'}", flush=True)
from google.colab import userdata
train_env = {
    **os.environ, "WANDB_API_KEY": userdata.get("WANDB_TOKEN"), "WANDB_MODE": "online",
    "WANDB_X_DISABLE_STATS": "false", "WANDB_X_STATS_SAMPLING_INTERVAL": "5",
}
if WANDB_ENTITY:
    train_env["WANDB_USERNAME"] = WANDB_ENTITY  # The installed RSL-RL writer reads this name.
command = [
    PYTHON, "-u", str(REPO_DIR / "training/scripts/train.py"), TASK,
    "--gpu-ids", "[0]",
    "--env.scene.num-envs", str(NUM_ENVS),
    "--agent.max-iterations", str(ITERATIONS),
    "--agent.num-steps-per-env", str(ROLLOUT_STEPS),
    "--agent.algorithm.num-mini-batches", str(NUM_MINIBATCHES),
    "--agent.save-interval", str(SAVE_INTERVAL),
    "--agent.logger", "wandb",
    "--agent.wandb-project", WANDB_PROJECT,
    "--agent.run-name", RUN_NAME,
]
if START_TERRAIN_LEVEL >= 0:
    command += ["--start-terrain-level", str(START_TERRAIN_LEVEL)]
if RESUME_CHECKPOINT:
    command += ["--from", RESUME_CHECKPOINT]
process = subprocess.Popen(
    command, cwd=WORK_DIR, env=train_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    process.send_signal(2)  # Forward Colab's interrupt to the trainer.
    process.wait()
    raise
if return_code:
    raise subprocess.CalledProcessError(return_code, command)


In [ ]:
# 6. Find recent checkpoints. Rerun cell 4 to select one for continuation.
checkpoints = sorted(WORK_DIR.glob("logs/rsl_rl/*/*/model_*.pt"), key=lambda p: p.stat().st_mtime)
for checkpoint in checkpoints[-10:]:
    print(checkpoint)
print("Each run folder also contains TensorBoard logs, policy.onnx, and params/ configs. Cloud metrics are in your W&B project.")
